# Export completed screen evidence (CPU only)

Blank RUN_ID selects the latest run. Downloads result tables, final images, and repair diagnostics after checking manifest hashes. No model download or inference. Attach the metrics ZIP first, then all image ZIPs; each is below 32 MiB. No GitHub token is needed.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import drive, files
import json, hashlib, re

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/KLTN_OCSD')
RUN_ID = "" #@param {type:"string"}
if not RUN_ID:
    RUN_ID = json.loads((ROOT / 'results/latest_run.json').read_text())['run_id']
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), 'Invalid RUN_ID'
manifest_path = ROOT / 'results' / 'runs' / RUN_ID / 'run_manifest.json'
manifest = json.loads(manifest_path.read_text())
assert manifest['status'] == 'complete', 'Run is incomplete: resume before interpreting results'
assert manifest['contract']['experiment']['tier'] in ('screen', 'repair_screen', 'semantic_screen')
methods = {'gligen', 'gligen_bon', 'grounded_repair', 'grounded_repair_no_m2', 'grounded_repair_sparse', 'semantic_repair', 'gligen_layout5', 'gligen_semantic5'}
selected = []
for relative, digest in manifest['artifacts'].items():
    rel = Path(relative)
    result = (rel.parts[:3] == ('results', 'runs', RUN_ID)
              and rel.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.log', '.png'})
    output = (rel.parts[:3] == ('outputs', 'runs', RUN_ID) and len(rel.parts) >= 6
              and rel.parts[3] == 'quickdraw' and rel.parts[4] in methods)
    if not (result or output):
        continue
    p = (ROOT / rel).resolve()
    assert p.is_relative_to(ROOT.resolve()), 'Invalid artifact path'
    assert p.is_file(), f'Missing: {p}'
    assert hashlib.sha256(p.read_bytes()).hexdigest() == digest, f'Changed: {p}'
    selected.append((p, relative))
assert selected, 'No artifacts found'
# Bound uncompressed size conservatively; each ZIP is independently readable.
MAX_PART_BYTES = 24 * 2**20
archives = []
def export_parts(label, entries):
    part, group = 1, []
    manifest_size = manifest_path.stat().st_size
    size = manifest_size
    def write_group(group, part):
        archive = f'/content/ocsd_review_{RUN_ID}_{label}_{part:02d}.zip'
        with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
            z.write(manifest_path, 'run_manifest.json')
            for p, relative in group:
                z.write(p, relative)
        assert Path(archive).stat().st_size < 32 * 2**20, 'Archive exceeds transfer limit'
        archives.append(archive)
    for p, relative in sorted(entries, key=lambda entry: entry[1]):
        item_size = p.stat().st_size + len(relative.encode('utf-8')) * 2 + 512
        assert manifest_size + item_size < MAX_PART_BYTES, f'Single artifact too large: {relative}'
        if group and size + item_size > MAX_PART_BYTES:
            write_group(group, part)
            part += 1
            group, size = [], manifest_size
        group.append((p, relative))
        size += item_size
    if group:
        write_group(group, part)

export_parts('metrics', [(p, rel) for p, rel in selected if p.suffix.lower() != '.png'])
export_parts('images', [(p, rel) for p, rel in selected if p.suffix.lower() == '.png'])
print('Exported existing files; no model execution:', len(selected), 'run:', RUN_ID)
print('Upload metrics first, then every image part. If browser blocks downloads, use Colab Files:')
for archive in archives:
    print(archive, round(Path(archive).stat().st_size / 2**20, 2), 'MiB')
    files.download(archive)